# 03 — Milestone I: full frozen comparison
Connect this notebook to a Colab GPU runtime and run from the top. It bootstraps a fresh kernel, prepares the locked data, trains both frozen backbones on all 4,040 training images for 40 epochs, evaluates all four test sets independently, and creates quantitative and qualitative comparisons.

In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = True

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch

project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)

bootstrap_env = os.environ.copy()
dino_weights = Path('/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
if not dino_weights.is_file():
    print('DINOv3 requires approved Meta access on the first run only.')
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
command = [sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
           '--project-dir', str(project_dir), '--state-file', str(state_file)]
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError('Review https://github.com/DengPingFan/SINet#9-license before accepting.')
command += ['--ensure-training-data', '--accept-noncommercial-license']
subprocess.run(command, cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = state['project_dir']
DATA_ROOT = state['data_root']
RUNS_ROOT = state['runs_root']
COMPARISONS_ROOT = state['comparisons_root']
SAMPLE_IMAGE = state['sample_image']
TRAIN_MANIFEST = state['train_manifest']
print('Ready on', state['gpu'])
EPOCHS = 40
QUALITATIVE_COUNT = 24
EXPERIMENT_TAG = 'phase1_seed42'  # Stable Drive names make interrupted runs resumable.
PROJECT_DIR = Path(PROJECT_DIR)
DATA_ROOT = Path(DATA_ROOT)
RUNS_ROOT = Path(RUNS_ROOT)
COMPARISONS_ROOT = Path(COMPARISONS_ROOT)
os.chdir(PROJECT_DIR)


In [ ]:
# Download/cache/extract CAMO-Test, COD10K-Test, CHAMELEON and NC4K; then build manifests.
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError('Review https://github.com/DengPingFan/SINet#9-license, then set the acceptance flag to True.')
subprocess.run([
    sys.executable, 'scripts/bootstrap_test_data.py',
    '--data-root', str(DATA_ROOT), '--manifest-dir', 'manifests',
    '--accept-noncommercial-license',
], check=True)
subprocess.run([sys.executable, 'scripts/validate_dataset.py', '--manifest-dir', 'manifests'], check=True)
os.sync()

In [ ]:
# Stable Drive paths make Colab disconnects recoverable. Completed runs are skipped.
DINO_RUN = RUNS_ROOT / f'{EXPERIMENT_TAG}_dinov3_vitb16'
VJEPA_RUN = RUNS_ROOT / f'{EXPERIMENT_TAG}_vjepa21_vitb16'
COMPARISON_DIR = COMPARISONS_ROOT / f'{EXPERIMENT_TAG}_frozen_comparison'
def train_or_resume(config, run_dir):
    last = run_dir / 'checkpoints' / 'last.pt'
    epoch_checkpoints = sorted((run_dir / 'checkpoints').glob('epoch_*.pt'))
    checkpoint = last if last.is_file() else (epoch_checkpoints[-1] if epoch_checkpoints else None)
    if checkpoint is not None:
        epoch = int(torch.load(checkpoint, map_location='cpu', weights_only=False)['epoch'])
        if epoch >= EPOCHS:
            print(f'Already complete ({epoch} epochs): {run_dir}')
            return
    command = [sys.executable, 'scripts/train.py', '--config', config,
               '--run-dir', str(run_dir), '--epochs', str(EPOCHS)]
    if checkpoint is not None:
        command += ['--resume', str(checkpoint)]
        print('Resuming from', checkpoint)
    subprocess.run(command, check=True)
    os.sync()
print('DINO_RUN =', DINO_RUN)
print('VJEPA_RUN =', VJEPA_RUN)
print('COMPARISON_DIR =', COMPARISON_DIR)

In [ ]:
# Full DINOv3 frozen run: all 4,040 images, locked Phase-1 configuration.
train_or_resume('configs/frozen_dinov3_vitb16.yaml', DINO_RUN)

In [ ]:
# Full V-JEPA 2.1 frozen run with the identical decoder and training protocol.
train_or_resume('configs/frozen_vjepa21_vitb16.yaml', VJEPA_RUN)

In [ ]:
# Evaluate each final checkpoint independently on all four locked test sets.
for run in (DINO_RUN, VJEPA_RUN):
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--run', str(run)], check=True)
os.sync()

In [ ]:
# Build metric/compute tables, comparison graphs, and paired qualitative overlays.
subprocess.run([sys.executable, 'scripts/compare_runs.py',
    str(DINO_RUN), str(VJEPA_RUN), '--output', str(COMPARISON_DIR),
    '--qualitative-count', str(QUALITATIVE_COUNT)], check=True)
os.sync()

In [ ]:
# Display the summary artifacts inline; all originals remain persisted in Drive.
from IPython.display import display
from PIL import Image
import pandas as pd
display(pd.read_csv(COMPARISON_DIR / 'comparison_metrics.csv'))
display(pd.read_csv(COMPARISON_DIR / 'compute_comparison.csv'))
display(Image.open(COMPARISON_DIR / 'metric_comparison.png'))
display(Image.open(COMPARISON_DIR / 'training_curves.png'))
panel_paths = sorted((COMPARISON_DIR / 'qualitative_panels').glob('*.png'))
for path in panel_paths[:8]:
    print(path.name); display(Image.open(path))
print(f'All {len(panel_paths)} panels:', COMPARISON_DIR / 'qualitative_panels')